# Predicción de Demanda - Sistema de Optimización Logística UPSE

Este notebook implementa modelos de Machine Learning para predecir la demanda futura de productos usando:
- **Prophet** de Facebook para series temporales
- **ARIMA** para análisis estadístico
- **Random Forest** para patrones complejos

In [ ]:
# Importar librerías necesarias
import sys
import os
sys.path.append(os.path.abspath('../backend'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# Configuración de visualización
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
%matplotlib inline

## 1. Cargar Datos desde la Base de Datos

In [ ]:
from sqlalchemy import create_engine
from sqlalchemy.orm import sessionmaker

# Conectar a la base de datos SQLite
DATABASE_URL = "sqlite:///../backend/logistic.db"
engine = create_engine(DATABASE_URL)
Session = sessionmaker(bind=engine)
session = Session()

print("✅ Conexión a base de datos establecida")

In [ ]:
# Cargar datos de demanda
query = """
SELECT 
    d.id,
    d.product_id,
    d.warehouse_id,
    d.date,
    d.quantity,
    d.forecast_quantity,
    p.name as product_name,
    p.category as product_category,
    w.name as warehouse_name,
    w.location as warehouse_location
FROM demands d
JOIN products p ON d.product_id = p.id
JOIN warehouses w ON d.warehouse_id = w.id
ORDER BY d.date
"""

df_demands = pd.read_sql(query, engine)
df_demands['date'] = pd.to_datetime(df_demands['date'])

print(f"📊 Datos cargados: {len(df_demands)} registros")
print(f"📅 Rango de fechas: {df_demands['date'].min()} a {df_demands['date'].max()}")
print(f"🏭 Bodegas: {df_demands['warehouse_name'].nunique()}")
print(f"📦 Productos: {df_demands['product_name'].nunique()}")
df_demands.head(10)

## 2. Análisis Exploratorio

In [ ]:
# Estadísticas descriptivas
print("📈 Estadísticas de Demanda:")
print(df_demands['quantity'].describe())

# Demanda por producto
print("\n📦 Demanda Total por Producto:")
demand_by_product = df_demands.groupby('product_name')['quantity'].sum().sort_values(ascending=False)
print(demand_by_product)

In [ ]:
# Visualización: Demanda total por producto
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# Gráfico 1: Demanda por producto
demand_by_product.plot(kind='bar', ax=axes[0, 0], color='steelblue')
axes[0, 0].set_title('Demanda Total por Producto', fontsize=14, fontweight='bold')
axes[0, 0].set_xlabel('Producto')
axes[0, 0].set_ylabel('Cantidad Total')
axes[0, 0].tick_params(axis='x', rotation=45)

# Gráfico 2: Demanda por bodega
demand_by_warehouse = df_demands.groupby('warehouse_name')['quantity'].sum().sort_values(ascending=False)
demand_by_warehouse.plot(kind='bar', ax=axes[0, 1], color='coral')
axes[0, 1].set_title('Demanda Total por Bodega', fontsize=14, fontweight='bold')
axes[0, 1].set_xlabel('Bodega')
axes[0, 1].set_ylabel('Cantidad Total')
axes[0, 1].tick_params(axis='x', rotation=45)

# Gráfico 3: Serie temporal de demanda agregada
daily_demand = df_demands.groupby('date')['quantity'].sum()
daily_demand.plot(ax=axes[1, 0], color='green', linewidth=2)
axes[1, 0].set_title('Demanda Diaria Agregada', fontsize=14, fontweight='bold')
axes[1, 0].set_xlabel('Fecha')
axes[1, 0].set_ylabel('Cantidad')
axes[1, 0].grid(True, alpha=0.3)

# Gráfico 4: Distribución de demanda
axes[1, 1].hist(df_demands['quantity'], bins=30, color='purple', alpha=0.7, edgecolor='black')
axes[1, 1].set_title('Distribución de Demanda', fontsize=14, fontweight='bold')
axes[1, 1].set_xlabel('Cantidad')
axes[1, 1].set_ylabel('Frecuencia')
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 3. Preparación de Datos para Modelos

In [ ]:
# Seleccionar un producto para el modelo ejemplo (el de mayor demanda)
top_product = demand_by_product.index[0]
print(f"🎯 Producto seleccionado para modelado: {top_product}")

# Filtrar datos del producto
df_product = df_demands[df_demands['product_name'] == top_product].copy()
df_product = df_product.groupby('date')['quantity'].sum().reset_index()
df_product = df_product.sort_values('date')

print(f"📊 Registros para modelado: {len(df_product)}")
df_product.head()

In [ ]:
# División train/test (80/20)
train_size = int(len(df_product) * 0.8)
df_train = df_product.iloc[:train_size]
df_test = df_product.iloc[train_size:]

print(f"🎓 Datos de entrenamiento: {len(df_train)} registros")
print(f"🧪 Datos de prueba: {len(df_test)} registros")

## 4. Modelo 1: Prophet de Facebook

In [ ]:
from prophet import Prophet

# Preparar datos para Prophet (requiere columnas 'ds' y 'y')
df_prophet = df_train.copy()
df_prophet.columns = ['ds', 'y']

# Crear y entrenar modelo
print("🤖 Entrenando modelo Prophet...")
model_prophet = Prophet(
    daily_seasonality=True,
    weekly_seasonality=True,
    yearly_seasonality=False,  # No tenemos un año completo de datos
    seasonality_mode='multiplicative',
    changepoint_prior_scale=0.05
)
model_prophet.fit(df_prophet)
print("✅ Modelo Prophet entrenado")

In [ ]:
# Hacer predicciones
future = model_prophet.make_future_dataframe(periods=len(df_test), freq='D')
forecast_prophet = model_prophet.predict(future)

# Visualizar predicciones
fig1 = model_prophet.plot(forecast_prophet, figsize=(14, 6))
plt.title(f'Predicción de Demanda con Prophet - {top_product}', fontsize=16, fontweight='bold')
plt.xlabel('Fecha')
plt.ylabel('Cantidad')
plt.grid(True, alpha=0.3)
plt.show()

# Componentes del modelo
fig2 = model_prophet.plot_components(forecast_prophet, figsize=(14, 8))
plt.show()

In [ ]:
# Evaluar modelo Prophet
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Obtener predicciones para el conjunto de prueba
test_predictions_prophet = forecast_prophet.iloc[train_size:train_size+len(df_test)]['yhat'].values
test_actual = df_test['quantity'].values

mae_prophet = mean_absolute_error(test_actual, test_predictions_prophet)
rmse_prophet = np.sqrt(mean_squared_error(test_actual, test_predictions_prophet))
r2_prophet = r2_score(test_actual, test_predictions_prophet)

print("📊 Métricas del Modelo Prophet:")
print(f"MAE (Error Absoluto Medio): {mae_prophet:.2f}")
print(f"RMSE (Raíz del Error Cuadrático Medio): {rmse_prophet:.2f}")
print(f"R² Score: {r2_prophet:.4f}")

## 5. Modelo 2: ARIMA

In [ ]:
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

# Análisis de autocorrelación
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
plot_acf(df_train['quantity'], lags=20, ax=axes[0])
plot_pacf(df_train['quantity'], lags=20, ax=axes[1])
axes[0].set_title('Autocorrelación (ACF)')
axes[1].set_title('Autocorrelación Parcial (PACF)')
plt.tight_layout()
plt.show()

In [ ]:
# Entrenar modelo ARIMA (p, d, q)
print("🤖 Entrenando modelo ARIMA...")
model_arima = ARIMA(df_train['quantity'], order=(2, 1, 2))
fitted_arima = model_arima.fit()
print("✅ Modelo ARIMA entrenado")
print(fitted_arima.summary())

In [ ]:
# Hacer predicciones
forecast_arima = fitted_arima.forecast(steps=len(df_test))

# Evaluar modelo ARIMA
mae_arima = mean_absolute_error(test_actual, forecast_arima)
rmse_arima = np.sqrt(mean_squared_error(test_actual, forecast_arima))
r2_arima = r2_score(test_actual, forecast_arima)

print("📊 Métricas del Modelo ARIMA:")
print(f"MAE (Error Absoluto Medio): {mae_arima:.2f}")
print(f"RMSE (Raíz del Error Cuadrático Medio): {rmse_arima:.2f}")
print(f"R² Score: {r2_arima:.4f}")

In [ ]:
# Visualizar predicciones ARIMA
plt.figure(figsize=(14, 6))
plt.plot(df_train['date'], df_train['quantity'], label='Entrenamiento', color='blue', linewidth=2)
plt.plot(df_test['date'], test_actual, label='Real (Test)', color='green', linewidth=2)
plt.plot(df_test['date'], forecast_arima, label='Predicción ARIMA', color='red', linestyle='--', linewidth=2)
plt.title(f'Predicción con ARIMA - {top_product}', fontsize=16, fontweight='bold')
plt.xlabel('Fecha')
plt.ylabel('Cantidad')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 6. Modelo 3: Random Forest

In [ ]:
from sklearn.ensemble import RandomForestRegressor

# Crear características temporales
def create_features(df):
    df = df.copy()
    df['day_of_week'] = df['date'].dt.dayofweek
    df['day_of_month'] = df['date'].dt.day
    df['month'] = df['date'].dt.month
    df['week_of_year'] = df['date'].dt.isocalendar().week
    
    # Lag features (valores anteriores)
    df['lag_1'] = df['quantity'].shift(1)
    df['lag_7'] = df['quantity'].shift(7)
    df['rolling_mean_7'] = df['quantity'].rolling(window=7, min_periods=1).mean()
    df['rolling_std_7'] = df['quantity'].rolling(window=7, min_periods=1).std()
    
    return df

# Preparar datos
df_ml = df_product.copy()
df_ml = create_features(df_ml)
df_ml = df_ml.dropna()  # Eliminar filas con NaN por los lags

# Dividir nuevamente después de crear features
train_size_ml = int(len(df_ml) * 0.8)
df_train_ml = df_ml.iloc[:train_size_ml]
df_test_ml = df_ml.iloc[train_size_ml:]

features = ['day_of_week', 'day_of_month', 'month', 'week_of_year', 'lag_1', 'lag_7', 'rolling_mean_7', 'rolling_std_7']
X_train = df_train_ml[features]
y_train = df_train_ml['quantity']
X_test = df_test_ml[features]
y_test = df_test_ml['quantity']

print(f"✅ Features creadas: {len(features)}")
print(f"📊 X_train shape: {X_train.shape}")
print(f"📊 X_test shape: {X_test.shape}")

In [ ]:
# Entrenar Random Forest
print("🤖 Entrenando modelo Random Forest...")
model_rf = RandomForestRegressor(
    n_estimators=100,
    max_depth=10,
    min_samples_split=5,
    random_state=42,
    n_jobs=-1
)
model_rf.fit(X_train, y_train)
print("✅ Modelo Random Forest entrenado")

In [ ]:
# Hacer predicciones
predictions_rf = model_rf.predict(X_test)

# Evaluar modelo
mae_rf = mean_absolute_error(y_test, predictions_rf)
rmse_rf = np.sqrt(mean_squared_error(y_test, predictions_rf))
r2_rf = r2_score(y_test, predictions_rf)

print("📊 Métricas del Modelo Random Forest:")
print(f"MAE (Error Absoluto Medio): {mae_rf:.2f}")
print(f"RMSE (Raíz del Error Cuadrático Medio): {rmse_rf:.2f}")
print(f"R² Score: {r2_rf:.4f}")

In [ ]:
# Importancia de características
feature_importance = pd.DataFrame({
    'feature': features,
    'importance': model_rf.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(10, 6))
plt.barh(feature_importance['feature'], feature_importance['importance'], color='teal')
plt.xlabel('Importancia')
plt.title('Importancia de Características - Random Forest', fontsize=14, fontweight='bold')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

In [ ]:
# Visualizar predicciones Random Forest
plt.figure(figsize=(14, 6))
plt.plot(df_test_ml['date'], y_test.values, label='Real (Test)', color='green', linewidth=2)
plt.plot(df_test_ml['date'], predictions_rf, label='Predicción RF', color='purple', linestyle='--', linewidth=2)
plt.title(f'Predicción con Random Forest - {top_product}', fontsize=16, fontweight='bold')
plt.xlabel('Fecha')
plt.ylabel('Cantidad')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 7. Comparación de Modelos

In [ ]:
# Tabla comparativa
comparison = pd.DataFrame({
    'Modelo': ['Prophet', 'ARIMA', 'Random Forest'],
    'MAE': [mae_prophet, mae_arima, mae_rf],
    'RMSE': [rmse_prophet, rmse_arima, rmse_rf],
    'R² Score': [r2_prophet, r2_arima, r2_rf]
})

print("\n🏆 COMPARACIÓN DE MODELOS:")
print("="*60)
print(comparison.to_string(index=False))
print("="*60)

# Identificar mejor modelo
best_model_idx = comparison['R² Score'].idxmax()
best_model = comparison.loc[best_model_idx, 'Modelo']
print(f"\n🥇 Mejor modelo: {best_model} (R² = {comparison.loc[best_model_idx, 'R² Score']:.4f})")

In [ ]:
# Visualización comparativa
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

metrics = ['MAE', 'RMSE', 'R² Score']
colors = ['skyblue', 'lightcoral', 'lightgreen']

for idx, metric in enumerate(metrics):
    comparison.plot(x='Modelo', y=metric, kind='bar', ax=axes[idx], color=colors[idx], legend=False)
    axes[idx].set_title(f'{metric} por Modelo', fontsize=12, fontweight='bold')
    axes[idx].set_ylabel(metric)
    axes[idx].set_xlabel('')
    axes[idx].tick_params(axis='x', rotation=0)
    
plt.tight_layout()
plt.show()

## 8. Guardar Modelos Entrenados

In [ ]:
import pickle
import json

# Crear directorio para modelos
models_dir = '../data/models'
os.makedirs(models_dir, exist_ok=True)

# Guardar modelo Prophet
with open(f'{models_dir}/prophet_model.pkl', 'wb') as f:
    pickle.dump(model_prophet, f)
print("✅ Modelo Prophet guardado")

# Guardar modelo ARIMA
fitted_arima.save(f'{models_dir}/arima_model.pkl')
print("✅ Modelo ARIMA guardado")

# Guardar modelo Random Forest
with open(f'{models_dir}/random_forest_model.pkl', 'wb') as f:
    pickle.dump(model_rf, f)
print("✅ Modelo Random Forest guardado")

# Guardar metadata
metadata = {
    'product': top_product,
    'training_date': datetime.now().isoformat(),
    'metrics': {
        'prophet': {'mae': mae_prophet, 'rmse': rmse_prophet, 'r2': r2_prophet},
        'arima': {'mae': mae_arima, 'rmse': rmse_arima, 'r2': r2_arima},
        'random_forest': {'mae': mae_rf, 'rmse': rmse_rf, 'r2': r2_rf}
    },
    'best_model': best_model,
    'features': features
}

with open(f'{models_dir}/models_metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)
print("✅ Metadata guardada")

print(f"\n📁 Modelos guardados en: {os.path.abspath(models_dir)}")

## 9. Generar Predicciones Futuras

In [ ]:
# Predicciones para los próximos 30 días usando el mejor modelo
days_to_predict = 30
last_date = df_product['date'].max()
future_dates = pd.date_range(start=last_date + timedelta(days=1), periods=days_to_predict, freq='D')

print(f"🔮 Generando predicciones para los próximos {days_to_predict} días...")
print(f"📅 Desde: {future_dates[0].date()} hasta: {future_dates[-1].date()}")

# Usar Prophet para predicciones futuras
future_prophet = model_prophet.make_future_dataframe(periods=len(df_product) + days_to_predict, freq='D')
forecast_future = model_prophet.predict(future_prophet)
future_predictions = forecast_future.tail(days_to_predict)[['ds', 'yhat', 'yhat_lower', 'yhat_upper']]
future_predictions.columns = ['date', 'predicted_quantity', 'lower_bound', 'upper_bound']

print("\n📊 Predicciones futuras (primeros 10 días):")
print(future_predictions.head(10))

In [ ]:
# Visualizar predicciones futuras
plt.figure(figsize=(14, 6))
plt.plot(df_product['date'], df_product['quantity'], label='Histórico', color='blue', linewidth=2)
plt.plot(future_predictions['date'], future_predictions['predicted_quantity'], 
         label='Predicción', color='red', linestyle='--', linewidth=2)
plt.fill_between(future_predictions['date'], 
                  future_predictions['lower_bound'], 
                  future_predictions['upper_bound'], 
                  alpha=0.2, color='red', label='Intervalo de confianza')
plt.axvline(x=last_date, color='gray', linestyle=':', linewidth=2, label='Hoy')
plt.title(f'Predicción de Demanda Futura - {top_product}', fontsize=16, fontweight='bold')
plt.xlabel('Fecha')
plt.ylabel('Cantidad')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 10. Actualizar Base de Datos con Predicciones

In [ ]:
from app.models.models import Demand

# Obtener product_id y warehouse_id
product_info = df_demands[df_demands['product_name'] == top_product].iloc[0]
product_id = int(product_info['product_id'])
warehouse_id = int(product_info['warehouse_id'])

print(f"📦 Actualizando predicciones para:")
print(f"   - Producto: {top_product} (ID: {product_id})")
print(f"   - Bodega: {product_info['warehouse_name']} (ID: {warehouse_id})")

# Crear registros de demanda con predicciones
new_demands = []
for _, row in future_predictions.iterrows():
    demand = Demand(
        product_id=product_id,
        warehouse_id=warehouse_id,
        date=row['date'].date(),
        quantity=0,  # Cantidad real aún desconocida
        forecast_quantity=max(0, int(row['predicted_quantity']))  # No puede ser negativa
    )
    new_demands.append(demand)

# Guardar en base de datos
try:
    session.bulk_save_objects(new_demands)
    session.commit()
    print(f"\n✅ {len(new_demands)} predicciones guardadas en la base de datos")
except Exception as e:
    session.rollback()
    print(f"❌ Error al guardar predicciones: {e}")
finally:
    session.close()

## 📝 Conclusiones

### Resultados del Entrenamiento:

1. **Modelos Entrenados**:
   - ✅ Prophet: Modelo de series temporales de Facebook
   - ✅ ARIMA: Modelo estadístico clásico
   - ✅ Random Forest: Modelo de Machine Learning con features temporales

2. **Métricas de Evaluación**:
   - Se evaluaron MAE, RMSE y R² Score para cada modelo
   - El mejor modelo fue seleccionado automáticamente

3. **Predicciones Generadas**:
   - Se generaron predicciones para los próximos 30 días
   - Las predicciones incluyen intervalos de confianza
   - Los resultados fueron guardados en la base de datos

4. **Modelos Guardados**:
   - Los modelos entrenados están en `data/models/`
   - Pueden ser cargados y usados para predicciones en tiempo real
   - La metadata incluye métricas de rendimiento

### Próximos Pasos:

- Integrar los modelos en la API FastAPI
- Crear endpoint `/api/analytics/forecast` para predicciones
- Implementar reentrenamiento automático periódico
- Extender el análisis a todos los productos y bodegas